# RDD Joins in PySpark

This notebook demonstrates various types of joins using RDD API in PySpark: inner join, left outer join, right outer join, and full outer join.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('RDD Joins') \
    .getOrCreate()

sc = spark.sparkContext
print(f"Spark Version: {spark.version}")

## Understanding RDD Joins

RDD joins work with key-value pairs (tuples) where:
- First element is the **key** (used for joining)
- Second element is the **value** (data associated with the key)

Format: `(key, value)`

### Join Types:
1. **Inner Join**: Returns only matching keys from both RDDs
2. **Left Outer Join**: All keys from left RDD + matching from right (None for non-matching)
3. **Right Outer Join**: All keys from right RDD + matching from left (None for non-matching)
4. **Full Outer Join**: All keys from both RDDs (None for non-matching)

## 1. Inner Join

### Basic Inner Join Example

In [ ]:
# Create sample RDDs with key-value pairs
sample_rdd1 = sc.parallelize([("a", 1), ("b", 4)])
sample_rdd2 = sc.parallelize([("a", 2), ("a", 3)])

print("RDD 1 (key, value):")
print(sample_rdd1.collect())

print("\nRDD 2 (key, value):")
print(sample_rdd2.collect())

# Perform inner join
result = sample_rdd1.join(sample_rdd2)

print("\nInner Join Result:")
print(result.collect())
print("\nExplanation: Key 'a' exists in both RDDs, creates all combinations")
print("Key 'b' only in RDD1, so excluded from result")

### Inner Join with More Data

In [ ]:
# Create RDDs with multiple keys
rdd1 = sc.parallelize([
    ("apple", 10),
    ("banana", 20),
    ("orange", 15),
    ("grape", 8)
])

rdd2 = sc.parallelize([
    ("apple", "red"),
    ("banana", "yellow"),
    ("grape", "purple"),
    ("mango", "yellow")  # This won't match
])

print("RDD1 (Fruit, Quantity):")
for item in rdd1.collect():
    print(f"  {item}")

print("\nRDD2 (Fruit, Color):")
for item in rdd2.collect():
    print(f"  {item}")

# Inner join
inner_result = rdd1.join(rdd2)

print("\nInner Join Result (Fruit, (Quantity, Color)):")
for item in inner_result.collect():
    fruit, (qty, color) = item
    print(f"  {fruit}: Quantity={qty}, Color={color}")

print("\nNote: 'orange' and 'mango' excluded (not in both RDDs)")

## 2. Left Outer Join

### Basic Left Outer Join

In [ ]:
# Create sample RDDs
sample_rdd1 = sc.parallelize([("a", 1), ("b", 4)])
sample_rdd2 = sc.parallelize([("a", 2)])

print("RDD 1:")
print(sample_rdd1.collect())

print("\nRDD 2:")
print(sample_rdd2.collect())

# Left outer join
result = sample_rdd1.leftOuterJoin(sample_rdd2)

print("\nLeft Outer Join Result:")
print(result.collect())
print("\nExplanation: All keys from left RDD kept")
print("Key 'b' has None for right value (no match in RDD2)")

### Left Outer Join with More Data

In [ ]:
# Employee and Department RDDs
employees = sc.parallelize([
    ("E001", "John"),
    ("E002", "Jane"),
    ("E003", "Bob"),
    ("E004", "Alice")
])

departments = sc.parallelize([
    ("E001", "IT"),
    ("E002", "HR"),
    ("E003", "Finance")
    # E004 has no department assignment
])

print("Employees (ID, Name):")
for item in employees.collect():
    print(f"  {item}")

print("\nDepartments (ID, Dept):")
for item in departments.collect():
    print(f"  {item}")

# Left outer join - keep all employees
result = employees.leftOuterJoin(departments)

print("\nLeft Outer Join Result (All Employees):")
for emp_id, (name, dept) in result.collect():
    dept_name = dept if dept else "Unassigned"
    print(f"  {emp_id}: {name} - {dept_name}")

print("\nNote: Alice shown with 'Unassigned' department")

## 3. Right Outer Join

### Basic Right Outer Join

In [ ]:
# Create sample RDDs
sample_rdd1 = sc.parallelize([("a", 1), ("b", 4)])
sample_rdd2 = sc.parallelize([("a", 2)])

print("RDD 1:")
print(sample_rdd1.collect())

print("\nRDD 2:")
print(sample_rdd2.collect())

# Right outer join
result = sample_rdd2.rightOuterJoin(sample_rdd1)

print("\nRight Outer Join Result:")
print(result.collect())
print("\nExplanation: All keys from right RDD (RDD1) kept")
print("Key 'b' has None for left value (no match in RDD2)")

### Right Outer Join with More Data

In [ ]:
# Products and Inventory
products = sc.parallelize([
    ("P001", "Laptop"),
    ("P002", "Mouse"),
    ("P003", "Keyboard")
])

inventory = sc.parallelize([
    ("P001", 50),
    ("P002", 200),
    ("P003", 100),
    ("P004", 75)  # Product P004 not in products list
])

print("Products (ID, Name):")
for item in products.collect():
    print(f"  {item}")

print("\nInventory (ID, Quantity):")
for item in inventory.collect():
    print(f"  {item}")

# Right outer join - keep all inventory items
result = products.rightOuterJoin(inventory)

print("\nRight Outer Join Result (All Inventory):")
for prod_id, (name, qty) in result.collect():
    prod_name = name if name else "Unknown Product"
    print(f"  {prod_id}: {prod_name} - Quantity: {qty}")

print("\nNote: P004 shown with 'Unknown Product' name")

## 4. Full Outer Join

### Basic Full Outer Join

In [ ]:
# Create sample RDDs
sample_rdd1 = sc.parallelize([("a", 1), ("b", 4)])
sample_rdd2 = sc.parallelize([("a", 2), ("c", 8)])

print("RDD 1:")
print(sample_rdd1.collect())

print("\nRDD 2:")
print(sample_rdd2.collect())

# Full outer join
result = sample_rdd1.fullOuterJoin(sample_rdd2)

print("\nFull Outer Join Result:")
print(result.collect())
print("\nExplanation: All keys from both RDDs included")
print("Key 'b' has None for right value")
print("Key 'c' has None for left value")

### Full Outer Join with More Data

In [ ]:
# Students and Grades
students = sc.parallelize([
    ("S001", "Alice"),
    ("S002", "Bob"),
    ("S003", "Charlie"),
    ("S004", "David")
])

grades = sc.parallelize([
    ("S001", "A"),
    ("S002", "B"),
    ("S005", "A"),  # Student not in students list
    ("S006", "C")   # Student not in students list
])

print("Students (ID, Name):")
for item in students.collect():
    print(f"  {item}")

print("\nGrades (ID, Grade):")
for item in grades.collect():
    print(f"  {item}")

# Full outer join - keep all students and all grades
result = students.fullOuterJoin(grades)

print("\nFull Outer Join Result (All Students and Grades):")
for student_id, (name, grade) in sorted(result.collect()):
    student_name = name if name else "Unknown"
    student_grade = grade if grade else "Not Graded"
    print(f"  {student_id}: {student_name} - Grade: {student_grade}")

print("\nNote: Shows students without grades AND grades without student records")

## Comparing All Join Types Side by Side

In [ ]:
# Create test RDDs
left_rdd = sc.parallelize([
    ("a", 1),
    ("b", 2),
    ("c", 3)
])

right_rdd = sc.parallelize([
    ("a", "apple"),
    ("b", "banana"),
    ("d", "date")
])

print("LEFT RDD:")
print(left_rdd.collect())

print("\nRIGHT RDD:")
print(right_rdd.collect())

print("\n" + "="*60)
print("INNER JOIN (only matching keys):")
print("="*60)
for item in left_rdd.join(right_rdd).collect():
    print(f"  {item}")

print("\n" + "="*60)
print("LEFT OUTER JOIN (all left keys, right None if no match):")
print("="*60)
for item in left_rdd.leftOuterJoin(right_rdd).collect():
    print(f"  {item}")

print("\n" + "="*60)
print("RIGHT OUTER JOIN (all right keys, left None if no match):")
print("="*60)
for item in left_rdd.rightOuterJoin(right_rdd).collect():
    print(f"  {item}")

print("\n" + "="*60)
print("FULL OUTER JOIN (all keys from both, None for non-matching):")
print("="*60)
for item in left_rdd.fullOuterJoin(right_rdd).collect():
    print(f"  {item}")

## Practical Example: Customer Orders

In [ ]:
# Customer information
customers = sc.parallelize([
    ("C001", "John Doe"),
    ("C002", "Jane Smith"),
    ("C003", "Bob Wilson"),
    ("C004", "Alice Brown")
])

# Customer orders
orders = sc.parallelize([
    ("C001", 299.99),
    ("C001", 149.50),
    ("C002", 399.00),
    ("C003", 89.99),
    ("C005", 199.00)  # Order from customer not in our database
])

print("Customers:")
for cust_id, name in customers.collect():
    print(f"  {cust_id}: {name}")

print("\nOrders:")
for cust_id, amount in orders.collect():
    print(f"  {cust_id}: ${amount}")

print("\n" + "="*70)
print("INNER JOIN - Only customers with orders:")
print("="*70)
inner = customers.join(orders)
for cust_id, (name, amount) in inner.collect():
    print(f"  {cust_id}: {name} ordered ${amount}")

print("\n" + "="*70)
print("LEFT OUTER JOIN - All customers (showing who hasn't ordered):")
print("="*70)
left = customers.leftOuterJoin(orders)
for cust_id, (name, amount) in left.collect():
    if amount:
        print(f"  {cust_id}: {name} ordered ${amount}")
    else:
        print(f"  {cust_id}: {name} - NO ORDERS")

print("\n" + "="*70)
print("FULL OUTER JOIN - All customers and all orders:")
print("="*70)
full = customers.fullOuterJoin(orders)
for cust_id, (name, amount) in full.collect():
    cust_name = name if name else "UNKNOWN CUSTOMER"
    order_info = f"${amount}" if amount else "NO ORDERS"
    print(f"  {cust_id}: {cust_name} - {order_info}")

## Aggregating After Joins

In [ ]:
# Calculate total orders per customer
print("Total Order Amount per Customer:")
print("="*50)

# Join and then aggregate
customer_totals = customers.leftOuterJoin(orders) \
    .mapValues(lambda x: x[1] if x[1] else 0) \
    .reduceByKey(lambda a, b: a + b) \
    .join(customers.map(lambda x: (x[0], x[1])))  # Add names back

for cust_id, (total, name) in sorted(customer_totals.collect(), key=lambda x: x[1][0], reverse=True):
    print(f"  {name} ({cust_id}): ${total:.2f}")

## Multiple Joins Example

In [ ]:
# Employees, Departments, and Locations
employees_rdd = sc.parallelize([
    ("E001", ("John", "D1")),
    ("E002", ("Jane", "D2")),
    ("E003", ("Bob", "D1"))
])

departments_rdd = sc.parallelize([
    ("D1", "IT"),
    ("D2", "HR"),
    ("D3", "Finance")
])

locations_rdd = sc.parallelize([
    ("D1", "New York"),
    ("D2", "San Francisco"),
    ("D3", "Chicago")
])

# Transform employees to have department as key
emp_by_dept = employees_rdd.map(lambda x: (x[1][1], (x[0], x[1][0])))

print("Employees by Department:")
for item in emp_by_dept.collect():
    print(f"  {item}")

# Join with departments
emp_dept = emp_by_dept.join(departments_rdd)

print("\nEmployees with Department Names:")
for dept_id, ((emp_id, emp_name), dept_name) in emp_dept.collect():
    print(f"  {emp_name} ({emp_id}) works in {dept_name} ({dept_id})")

# Join with locations
emp_dept_loc = emp_dept.join(locations_rdd)

print("\nComplete Information (Employee + Department + Location):")
for dept_id, (((emp_id, emp_name), dept_name), location) in emp_dept_loc.collect():
    print(f"  {emp_name} | {dept_name} | {location}")

## Performance Considerations

In [ ]:
import time

# Create larger RDDs for performance testing
large_rdd1 = sc.parallelize([(i, i*10) for i in range(10000)])
large_rdd2 = sc.parallelize([(i, i*100) for i in range(5000, 15000)])

print("Performance comparison with larger datasets:")
print(f"RDD1 size: {large_rdd1.count()} records")
print(f"RDD2 size: {large_rdd2.count()} records")

# Inner join
start = time.time()
inner_count = large_rdd1.join(large_rdd2).count()
inner_time = time.time() - start
print(f"\nInner Join: {inner_count} records in {inner_time:.4f}s")

# Left outer join
start = time.time()
left_count = large_rdd1.leftOuterJoin(large_rdd2).count()
left_time = time.time() - start
print(f"Left Outer Join: {left_count} records in {left_time:.4f}s")

# Full outer join
start = time.time()
full_count = large_rdd1.fullOuterJoin(large_rdd2).count()
full_time = time.time() - start
print(f"Full Outer Join: {full_count} records in {full_time:.4f}s")

print("\nNote: Full outer join typically slower as it processes all records from both sides")

## Key Takeaways

### Join Types Summary:

| Join Type | Returns | Use Case |
|-----------|---------|----------|
| **Inner** | Only matching keys | When you only need matching records |
| **Left Outer** | All left + matching right | Keep all left records, optional right |
| **Right Outer** | All right + matching left | Keep all right records, optional left |
| **Full Outer** | All keys from both | Need all records from both sides |

### Result Format:
- **Inner**: `(key, (left_value, right_value))`
- **Left Outer**: `(key, (left_value, right_value_or_None))`
- **Right Outer**: `(key, (left_value_or_None, right_value))`
- **Full Outer**: `(key, (left_or_None, right_or_None))`

### Best Practices:
1. **Use Inner Join** when you only need matching records
2. **Use Left/Right Outer** when one dataset is primary
3. **Use Full Outer** when you need complete data from both sides
4. **Handle None values** when using outer joins
5. **Consider partitioning** for large datasets
6. **Cache frequently used RDDs** before multiple joins

### Performance Tips:
- Smaller RDD should be on the right for broadcasts
- Use `partitionBy()` before joins if doing multiple operations
- Inner joins are fastest (less data to process)
- Full outer joins are slowest (most data to process)

In [ ]:
# Stop Spark Session
spark.stop()